# Assignment 4 · CIFAR-10 Training
Yulin Zhang

Select a GPU runtime, then run the cells in order. Upload the Assignment 4 project ZIP in the first code cell.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from tempfile import mkdtemp
from google.colab import files
import torch

assert torch.cuda.is_available(), "Select a GPU in Runtime > Change runtime type."
uploaded = files.upload()
zip_name = next(iter(uploaded))
assert zip_name.lower().endswith(".zip"), "Upload the Assignment 4 ZIP."
project = Path(mkdtemp(prefix="assignment4_", dir="/content"))
with ZipFile(zip_name) as archive:
    archive.extractall(project)
%cd {project}
print(torch.cuda.get_device_name(0))

In [ ]:
# Download the public CIFAR-10 training split from the uoft-cs dataset mirror.
!pip -q install pyarrow
import subprocess
subprocess.run(["curl", "-fL", "--retry", "3",
    "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/train-00000-of-00001.parquet",
    "-o", "/content/cifar10-train.parquet"], check=True)
assert Path("/content/cifar10-train.parquet").stat().st_size > 100_000_000

In [ ]:
import subprocess
import sys

epochs = 20
for model in ("diffusion", "ebm"):
    subprocess.run([
        sys.executable, "train_advanced.py", model,
        "--epochs", str(epochs), "--parquet", "/content/cifar10-train.parquet",
    ], check=True)

In [ ]:
from IPython.display import Image, display

for model in ("diffusion", "ebm"):
    display(Image(filename=f"outputs/{model}_samples.png"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "verify_api.py"], check=True)

In [ ]:
from zipfile import ZIP_DEFLATED

out = "/content/Assignment4_Yulin_Zhang_Trained.zip"
with ZipFile(out, "w", ZIP_DEFLATED) as archive:
    for path in sorted(Path(".").rglob("*")):
        if path.is_file() and not {"data", "__pycache__", ".git"}.intersection(path.parts):
            if not path.name.endswith("_resume.pt"):
                archive.write(path, path.as_posix())
files.download(out)